# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p) (FAIR^2 dataset) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package via Croissant library
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")

# Optionally, view all metadata fields:
# pprint.pprint(metadata.to_json())

## 2. Data Overview

Review available record sets and their fields, including their `@id`s (unique identifiers in the Croissant schema).

Using `dataset.list_record_sets()` will help you discover record sets and their field ids.

In [ ]:
# List available record sets
record_sets = dataset.list_record_sets()
print("Available record sets:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name','(no name)')}")

# For this dataset, there may be only one core RecordSet. Let's view the fields/columns it contains.
for rs in record_sets:
    print(f"\nRecord Set: {rs.get('name','(no name)')}, @id: {rs['@id']}")
    print("  Fields (@id):")
    for field in rs.get('field', []):
        print(f"    - {field['@id']}: {field.get('name', '(no name)')}")
    print("  Columns (@id):")
    for column in rs.get('column', []):
        print(f"    - {column['@id']}: {column.get('name', '(no name)')}")

## 3. Data Extraction

Let's load data from the main record set (using its `@id`) into a pandas DataFrame for analysis.

*Note: Replace the record set and field `@id`s below with those you discover above if they change in the future. We'll assume the first record set is the core clinical table.*

In [ ]:
# Get the main (first) RecordSet @id
main_record_set_id = record_sets[0]['@id']
print(f"Using main record set: {main_record_set_id}")

# Optionally, list its fields and columns for later reference
field_ids = [field['@id'] for field in record_sets[0].get('field',[])]
column_ids = [col['@id'] for col in record_sets[0].get('column',[])]
print("Field @ids:", field_ids)
print("Column @ids:", column_ids)

# Extract the records from the main RecordSet
records = list(dataset.records(record_set=main_record_set_id))

# Turn into a DataFrame
df = pd.DataFrame(records)

# Show available columns (these correspond to field or column @ids)
print("Loaded DataFrame columns (as @id's):")
print(df.columns.tolist())

# Preview the data
df.head()

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate common processing steps:
- Filtering records on a numeric field (e.g., patient age)
- Normalizing the numeric field
- Grouping data (e.g., by anatomical location or sex)

# **IMPORTANT:**
* All fields must be referenced by their `@id` exactly as in the overview above (you can map them to friendlier variable names if desired, but always use @id in code).*

In [ ]:
# For demonstration, let's try to select a numeric field for filtering (e.g., Age at Diagnosis)
# Please inspect the field/column ids from the previous step and set this to the appropriate @id.
# Here we use a typical id found in clinical datasets, such as 'age_at_diagnosis' or similar.

# Example: Find a column containing 'age'. Adjust this to match your dataset!
age_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        age_field_id = col
        break

if not age_field_id:
    print("Could not find an age field in the DataFrame columns. Please inspect column names and adjust the code.")
else:
    print(f"Using age field: {age_field_id}")

    # Convert to numeric if needed
    df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')

    # Filter records: for example, only patients older than 40
    age_threshold = 40
    filtered_df = df[df[age_field_id] > age_threshold].copy()
    print(f"Filtered records with {age_field_id} > {age_threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize age field
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"Normalized {age_field_id} for filtered records:")
    display(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

    # Grouping: By anatomical_site or sex (find a suitable group field)
    group_field = None
    for col in df.columns:
        if 'sex' in col.lower() or 'gender' in col.lower():
            group_field = col
            break
    if not group_field:
        for col in df.columns:
            if 'anatomical' in col.lower():
                group_field = col
                break
    
    if group_field:
        print(f"\nGrouping data by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[age_field_id].mean()
        print(grouped_df)
    else:
        print("Could not find a grouping field such as 'sex', 'gender', or 'anatomical_site'.")

## 5. Visualization

Visualize the age distribution and sample grouping (e.g., by anatomical location or sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if age_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[age_field_id], kde=True, bins=15)
    plt.title(f'Distribution of Age ({age_field_id}) in Dataset')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

    # If we have a group field as above, do a boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=age_field_id)
        plt.title(f'Age by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel('Age')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have demonstrated loading, exploring, and performing preliminary analysis on the FAIR^2 clinical dataset using the `mlcroissant` library. All direct references to data entities (record sets, fields) were made using their `@id` for full reproducibility. You are now equipped to perform more detailed exploration and modeling on this and other ML-Croissant datasets.

- **Data loaded:** See all columns and sample rows above.
- **Basic EDA:** Filtered, normalized, and grouped by fields referenced via `@id`.
- **Plots:** Age distribution and group comparison visualization.

**Next steps:**
- Explore more fields using their `@id`, try different filtering/grouping criteria, or link to additional metadata as needed using the Croissant schema!